In [1]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

# Data Preprocessing

In [2]:
# Load datasets
passage_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [3]:
# Check the dataframes
print(passage_df.columns)
print(test_df.columns)
passage_df.head()

Index(['passage'], dtype='object')
Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [4]:
passage_df.shape

(40221, 1)

In [5]:
# Check test dataframe
test_df.head()

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."


In [6]:
test_df.shape

(4719, 3)

## Drop missing values

In [7]:
if 'text' in passage_df.columns:
    passage_df.dropna(subset=['text'], inplace=True)
if 'question' in test_df.columns:
    test_df.dropna(subset=['question'], inplace=True)

## Cleaning Function

In [8]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'<.*?>', '', text)  # Remove HTML tags if any
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
        text = text.strip()
    return text

if 'passage' in passage_df.columns:
    passage_df['clean_passage'] = passage_df['passage'].apply(clean_text)
if 'question' in test_df.columns:
    test_df['clean_question'] = test_df['question'].apply(clean_text)
if 'answer' in test_df.columns:
    test_df['clean_answer'] = test_df['answer'].apply(clean_text)

## Chunking for retrieval

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_TOKENS = min(256, tokenizer.model_max_length)
OVERLAP = 40

def chunk_passage(text, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    tokens = tokenizer.tokenize(text)
    chunks = []
    idx = 0
    while idx < len(tokens):
        chunk = tokens[idx : idx + max_tokens]
        chunk = tokenizer.convert_tokens_to_string(chunk)
        chunks.append(chunk)
        idx += max_tokens - overlap
    return chunks

In [10]:
if 'clean_passage' in passage_df.columns:
    passage_df['chunks'] = passage_df['clean_passage'].apply(lambda txt: chunk_passage(txt) if pd.notnull(txt) else [])

Token indices sequence length is longer than the specified maximum sequence length for this model (557 > 512). Running this sequence through the model will result in indexing errors


## Flatten to dataframe

In [11]:
chunked_passages = []
for idx, row in passage_df.iterrows():
    if isinstance(row['chunks'], list):
        for c_idx, chunk in enumerate(row['chunks']):
            chunked_passages.append({
                'doc_id': row.name,
                'chunk_id': f"{row.name}_{c_idx}",
                'chunk': chunk
            })
chunked_df = pd.DataFrame(chunked_passages)

## QA pairs for transformer finetuning

In [12]:
if {'clean_question', 'clean_answer'}.issubset(test_df.columns):
    qa_pairs = test_df[['clean_question', 'clean_answer']].dropna()
    qa_pairs = qa_pairs.rename(columns={'clean_question': 'question', 'clean_answer': 'answer'})
else:
    qa_pairs = pd.DataFrame(columns=['question', 'answer'])

## Train/validation split

In [13]:
if len(qa_pairs) > 1:
    train_set, val_set = train_test_split(qa_pairs, test_size=0.2, random_state=42)
else:
    train_set, val_set = qa_pairs, qa_pairs

In [14]:
# Save to disk
chunked_df.to_csv("chunked_passages.csv", index=False)
train_set.to_csv("train_set.csv", index=False)
val_set.to_csv("val_set.csv", index=False)

In [16]:
# Save tokenizer
tokenizer.save_pretrained("./bert-base-uncased-tokenizer")
print("Tokenizer saved")

Tokenizer saved


## Summary

In [15]:
print(f"Total passage chunks: {len(chunked_df)}")
print(f"Training examples: {len(train_set)} | Validation examples: {len(val_set)}")

Total passage chunks: 64395
Training examples: 3775 | Validation examples: 944


The train_set and val_set were split only from QA pairs extracted from `test_df`, since only it had both question and answers fields.

The `passage_df` lacks complete QA info, so including it would produce empty or invalid examples—not useful for training.

In [91]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, get_linear_schedule_with_warmup
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from lime.lime_text import LimeTextExplainer


In [93]:
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.qs = df['query'].tolist()
        self.ps = df['passage'].tolist()
        self.labels = df['label'].tolist()  # 0/1 or multi-class
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(self.qs[idx], self.ps[idx],
                       max_length=self.max_len, 
                       truncation=True,
                       return_tensors='pt')
        return {
            'input_ids': enc.input_ids.squeeze(0),
            'attention_mask': enc.attention_mask.squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [95]:
class CausalTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8,
                 num_layers=4, max_rel_pos=128,
                 pretrained_emb=None):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        if pretrained_emb is not None:
            self.embed.weight.data.copy_(pretrained_emb)

        self.pos_embed = nn.Embedding(2 * max_rel_pos + 1, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dropout=0.1,
            ) for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, 2)  # binary classification

    def forward(self, input_ids, attn_mask):
        embeddings = self.embed(input_ids)
        seq_len = input_ids.size(1)
        idxs = torch.arange(seq_len, device=input_ids.device)
        rel_pos = idxs[None, :] - idxs[:, None]
        rel_pos_clamped = rel_pos.clamp(-self.pos_embed.num_embeddings // 2,
                                        self.pos_embed.num_embeddings // 2)
        rel_pos_idxs = rel_pos_clamped + self.pos_embed.num_embeddings // 2
        pos_emb = self.pos_embed(rel_pos_idxs)
        x = embeddings + pos_emb.mean(dim=1)  # aggregate per-token
        for layer in self.layers:
            x = layer(x.permute(1, 0, 2), src_key_padding_mask=~attn_mask.bool()).permute(1, 0, 2)
        cls_repr = x[:, 0, :]
        return self.classifier(cls_repr)


In [97]:
def train(model, loader, opt, sched, device):
    model.train()
    total_loss = 0
    for b in loader:
        ids, mask, lbl = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
        out = model(ids, mask)
        loss = nn.CrossEntropyLoss()(out, lbl)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        total_loss += loss.item()
    return total_loss / len(loader)


In [99]:
def evaluate(model, loader, device):
    model.eval()
    ys, preds = [], []
    all_pairs = []
    for b in loader:
        ids, mask, lbl = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
        out = model(ids, mask)
        prob = torch.softmax(out, dim=1)[:,1].detach().cpu().numpy()
        preds += prob.tolist()
        ys += lbl.cpu().tolist()
        for q, p, pr in zip(b['query'], b['passage'], prob):
            all_pairs.append((q,p,round(pr,3)))
    auc = roc_auc_score(ys, preds)
    return auc, all_pairs


## Causal Transformer based models using pretrained word embeddings and position embeddings with relative positions -Shahnaz

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.metrics import accuracy_score
from tokenizers import Tokenizer
import numpy as np


In [64]:
# --- Hyperparameters ---
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 25
D_MODEL = 128
HEADS = 4
NUM_LAYERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# --- Load tokenizer ---
tokenizer = Tokenizer.from_file("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\tokenizer.json")
VOCAB_SIZE = tokenizer.get_vocab_size()

In [128]:
# Dataset for seq2seq generation
class QA_Dataset(Dataset):
    def __init__(self, questions, answers):
        self.inputs = []
        self.targets = []
        for q, a in zip(questions, answers):
            q_ids = tokenizer.encode(q).ids
            a_ids = tokenizer.encode(a).ids

            q_ids = q_ids[:MAX_LEN]
            a_ids = a_ids[:MAX_LEN - 1]  # Leave space for <eos>

            input_ids = q_ids
            target_ids = a_ids + [tokenizer.token_to_id("[EOS]") if "[EOS]" in tokenizer.get_vocab() else 0]

            # Pad input and target
            input_ids = input_ids + [0] * (MAX_LEN - len(input_ids))
            target_ids = target_ids + [0] * (MAX_LEN - len(target_ids))

            self.inputs.append(torch.tensor(input_ids))
            self.targets.append(torch.tensor(target_ids))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Relative Positional Embedding
class RelativePositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        self.rel_pos_embed = nn.Embedding(2 * max_len - 1, d_model)
        print("Relative position min/max:", rel_pos.min().item(), rel_pos.max().item())
        assert rel_pos.min() >= 0
        assert rel_pos.max() < self.rel_pos_embed.num_embeddings


    def forward(self, qlen, klen):
        device = self.rel_pos_embed.weight.device  # Ensures correct device
        range_q = torch.arange(qlen, device=device)[:, None]
        range_k = torch.arange(klen, device=device)[None, :]
        rel_pos = range_q - range_k + klen - 1  # Shift to be positive
        return self.rel_pos_embed(rel_pos)  # (qlen, klen, heads * head_dim)    

# Transformer block with causal attention + relative position embedding
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.dk = d_model // heads
        self.heads = heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.rel_pos_embed = RelativePositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        B, T, _ = x.size()
        q = self.q_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)

        rel_pos = self.rel_pos_embed(T, T).to(x.device)
        rel_pos = rel_pos.view(T, T, self.heads, self.dk).permute(2, 0, 1, 3)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores += torch.einsum("bhtd,htsd->bhts", q, rel_pos)
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores / np.sqrt(self.dk), dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)

        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x

# Full transformer model for sequence generation
class CausalTransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.Sequential(*[
            CausalTransformerBlock(d_model, heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.lm_head(x)  # logits


In [54]:
# --- Load CSVs ---
train_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\train_set.csv")
test_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\val_set.csv")



In [56]:
train_df.head(), test_df.head()

(                                            question  \
 0  which tools have been developed for computing ...   
 1  describe the application of whole genome seque...   
 2  why does cranberry juice help combat urinary t...   
 3  what percentage of rheumatoid arthritis patien...   
 4                  is the protein mcl1 antiapoptotic   
 
                                               answer  
 0  splitnetworks are a generalization of phylogen...  
 1  genetic testing is an important component of d...  
 2  cranberry products affect the surface properti...  
 3  despite this a substantial proportion of patie...  
 4               yes mcl1 is an antiapoptotic protein  ,
                                             question  \
 0  what are the effects of homozygosity of ednrb ...   
 1  what is the function of calciumsensing recepto...   
 2                        what are commensal bacteria   
 3            what are the ernaproducing centers epcs   
 4  does radiotherapy for prostate

In [58]:
train_dataset = QA_Dataset(train_df["question"].tolist(), train_df["answer"].tolist())
test_dataset = QA_Dataset(test_df["question"].tolist(), test_df["answer"].tolist())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [60]:
# Model
model = CausalTransformerGenerator(VOCAB_SIZE, D_MODEL, HEADS, NUM_LAYERS).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore padding
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [66]:
# Training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)  # (B, T, V)
        loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss / len(train_loader):.4f}")


Epoch 1/25 Loss: 6.8018
Epoch 2/25 Loss: 6.6148
Epoch 3/25 Loss: 6.4190
Epoch 4/25 Loss: 6.2078
Epoch 5/25 Loss: 6.0071
Epoch 6/25 Loss: 5.8071
Epoch 7/25 Loss: 5.6241
Epoch 8/25 Loss: 5.4605
Epoch 9/25 Loss: 5.3130
Epoch 10/25 Loss: 5.1817
Epoch 11/25 Loss: 5.0591
Epoch 12/25 Loss: 4.9474
Epoch 13/25 Loss: 4.8576
Epoch 14/25 Loss: 4.7511
Epoch 15/25 Loss: 4.6750
Epoch 16/25 Loss: 4.5915
Epoch 17/25 Loss: 4.5278
Epoch 18/25 Loss: 4.4533
Epoch 19/25 Loss: 4.3894
Epoch 20/25 Loss: 4.3227
Epoch 21/25 Loss: 4.2765
Epoch 22/25 Loss: 4.2217
Epoch 23/25 Loss: 4.1683
Epoch 24/25 Loss: 4.1316
Epoch 25/25 Loss: 4.0876


In [88]:
# Inference Function
def generate_answer(question, max_len=20):
    input_ids = tokenizer.encode(question).ids
    input_ids = input_ids[:MAX_LEN] + [0] * (MAX_LEN - len(input_ids))

    vocab_size = model.embedding.num_embeddings  # ✅ correct way to get vocab size
    input_ids = [min(tok, vocab_size - 1) for tok in input_ids]  # clip invalid tokens

    input_tensor = torch.tensor(input_ids, device=DEVICE).unsqueeze(0)

    for _ in range(max_len):
        with torch.no_grad():
            output = model(input_tensor)
        next_token = output[0, -1].argmax().item()
        input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]], device=DEVICE)], dim=1)

    return tokenizer.decode(input_tensor[0].tolist())
    
from sklearn.preprocessing import LabelEncoder
    
label_encoder = LabelEncoder()
    
decoded_preds = label_encoder.inverse_transform(all_preds)
decoded_labels = label_encoder.inverse_transform(all_labels)


# Example generation
print(generate_answer("What is deep learning?"))

NameError: name 'all_preds' is not defined

In [110]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [130]:
from sklearn.preprocessing import LabelEncoder

# Fit LabelEncoder on answers (once before training or inference)
label_encoder = LabelEncoder()
label_encoder.fit(train_df["answer"])  # Fit using training data

# Example decoding predictions
# These must be integer-encoded predictions from model output
# Example placeholder (replace with your actual predictions):
# all_preds = [0, 1, 2]
# all_labels = [0, 1, 2]

# decoded_preds = label_encoder.inverse_transform(all_preds)
# decoded_labels = label_encoder.inverse_transform(all_labels)
import torch

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
# Inference Function
def generate_answer(question, max_len=20):
    try:
        input_ids = tokenizer.encode(question).ids
        input_ids = input_ids[:MAX_LEN] + [0] * (MAX_LEN - len(input_ids))

        vocab_size = model.embedding.num_embeddings
        input_ids = [min(tok, vocab_size - 1) for tok in input_ids]

        input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)

        for _ in range(max_len):
            with torch.no_grad():
                output = model(input_tensor)  # output: [batch_size, seq_len, vocab_size]
            next_token = output[0, -1].argmax().item()

            # Stop if padding or EOS token is generated
            if next_token in [0, 102]:  # 0=PAD, 102=[SEP] for BERT
                break

            # Check max input length to prevent position overflow
            if input_tensor.shape[1] >= MAX_LEN:
                break

            next_token_tensor = torch.tensor([[next_token]], dtype=torch.long, device=DEVICE)
            input_tensor = torch.cat([input_tensor, next_token_tensor], dim=1)

        return tokenizer.decode(input_tensor[0].tolist(), skip_special_tokens=True)

    except Exception as e:
        print("🔥 Exception:", e)


# Example generation
print(generate_answer("What is deep learning?"))


🔥 Exception: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

None


In [98]:
print("Tokenizer vocab size:", tokenizer.get_vocab_size())


Tokenizer vocab size: 30522


In [100]:
print("Model vocab size:", model.embedding.num_embeddings)


Model vocab size: 30522


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import BertTokenizer

# --- Hyperparameters ---
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5  # Reduced for testing
D_MODEL = 128
HEADS = 4
NUM_LAYERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Load BERT tokenizer ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased-tokenizer")
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

# --- Dataset with proper tokenization ---
class QA_Dataset(Dataset):
    def __init__(self, questions, answers, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.inputs = []
        self.targets = []
        
        for q, a in zip(questions, answers):
            # Tokenize with BERT-style formatting
            encoded = tokenizer.encode_plus(
                text=q,
                text_pair=a,
                max_length=max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids = encoded['input_ids'].squeeze(0)
            token_type_ids = encoded['token_type_ids'].squeeze(0)
            
            # Create labels (shift next-token prediction)
            labels = input_ids.clone()
            # Mask non-answer parts (only compute loss on answer)
            labels[token_type_ids == 0] = -100
            # Shift for causal prediction
            labels[:-1] = labels[1:].clone()
            labels[-1] = -100
            
            self.inputs.append(input_ids)
            self.targets.append(labels)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# --- Fixed Positional Embedding ---
class RelativePositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.rel_pos_embed = nn.Embedding(2 * max_len - 1, d_model)

    def forward(self, qlen, klen):
        device = self.rel_pos_embed.weight.device
        range_vec = torch.arange(qlen, device=device)
        distance_mat = range_vec[:, None] - range_vec[None, :]
        distance_mat_clipped = torch.clamp(
            distance_mat + self.max_len - 1, 
            min=0, 
            max=2 * self.max_len - 2
        )
        return self.rel_pos_embed(distance_mat_clipped)

# --- Transformer Block ---
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.dk = d_model // heads
        self.heads = heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rel_pos_embed = RelativePositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, padding_mask=None):
        B, T, _ = x.size()
        q = self.q_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.dk)
        
        # Add relative positions
        rel_pos = self.rel_pos_embed(T, T).to(x.device)
        rel_pos = rel_pos.view(T, T, self.heads, self.dk).permute(2, 0, 1, 3)
        scores += torch.einsum("bhtd,htsd->bhts", q, rel_pos)
        
        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(1), float('-inf'))
        
        # Apply padding mask if provided
        if padding_mask is not None:
            padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)
            scores = scores.masked_fill(~padding_mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        
        # Residual connections
        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x

# --- Generator Model ---
class CausalTransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.ModuleList([
            CausalTransformerBlock(d_model, heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        print(f"Model vocab size: {vocab_size}")

    def forward(self, x, padding_mask=None):
        x = self.embedding(x)
        for layer in self.transformer:
            x = layer(x, padding_mask)
        return self.lm_head(x)

# --- Load Data ---
train_df = pd.read_csv("train_set.csv")
test_df = pd.read_csv("val_set.csv")

# Handle missing data
train_df = train_df.dropna(subset=['question', 'answer'])
test_df = test_df.dropna(subset=['question', 'answer'])

# Initialize datasets
train_dataset = QA_Dataset(
    train_df["question"].tolist(),
    train_df["answer"].tolist(),
    tokenizer,
    MAX_LEN
)
test_dataset = QA_Dataset(
    test_df["question"].tolist(),
    test_df["answer"].tolist(),
    tokenizer,
    MAX_LEN
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# --- Initialize Model ---
model = CausalTransformerGenerator(VOCAB_SIZE, D_MODEL, HEADS, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore masked tokens

# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        # Create padding mask
        padding_mask = (inputs != tokenizer.pad_token_id).float().to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs, padding_mask=padding_mask)
        loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

# --- Generation Function ---
def generate_answer(question, max_len=30, temperature=0.9):
    model.eval()
    # Tokenize question
    inputs = tokenizer.encode_plus(
        question,
        max_length=MAX_LEN,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    
    # Generate tokens
    for _ in range(max_len):
        with torch.no_grad():
            outputs = model(input_ids, padding_mask=attention_mask)
        
        # Get last token logits
        logits = outputs[0, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        
        # Stop if [SEP] is generated
        if next_token.item() == tokenizer.sep_token_id:
            break
            
        # Update inputs
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        attention_mask = torch.cat([
            attention_mask, 
            torch.ones(1, 1, dtype=attention_mask.dtype, device=DEVICE)
        ], dim=1)
    
    # Decode and clean output
    output_text = tokenizer.decode(
        input_ids[0], 
        skip_special_tokens=True
    )
    # Remove question part
    return output_text.replace(question, "").strip()

# --- Test Generation ---
print("\nGenerated Answer:")
print(generate_answer("What is deep learning?"))

Using device: cuda


OSError: Can't load tokenizer for './bert-base-uncased-tokenizer'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './bert-base-uncased-tokenizer' is the correct path to a directory containing all relevant files for a BertTokenizer tokenizer.

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel

# --- Hyperparameters ---
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5  # Reduced for testing
D_MODEL = 128
HEADS = 4
NUM_LAYERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Load standard BERT tokenizer ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

# --- Simplified Dataset ---
class QA_Dataset(Dataset):
    def __init__(self, questions, answers, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.questions = questions
        self.answers = answers
        
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        question = str(self.questions[idx])
        answer = str(self.answers[idx])
        
        # Tokenize with question-answer format
        encoding = self.tokenizer(
            question,
            answer,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        token_type_ids = encoding['token_type_ids'].squeeze(0)
        
        # Create labels (shift next-token prediction)
        labels = input_ids.clone()
        # Only compute loss on answer part (token_type_ids=1)
        labels[token_type_ids == 0] = -100
        # Shift labels for next-token prediction
        labels[:-1] = labels[1:].clone()
        labels[-1] = -100
        
        return input_ids, token_type_ids, labels

# --- Fixed Positional Embedding ---
class RelativePositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.rel_pos_embed = nn.Embedding(2 * max_len - 1, d_model)

    def forward(self, qlen, klen):
        device = self.rel_pos_embed.weight.device
        range_vec = torch.arange(qlen, device=device)
        distance_mat = range_vec[:, None] - range_vec[None, :]
        distance_mat_clipped = torch.clamp(
            distance_mat + self.max_len - 1, 
            min=0, 
            max=2 * self.max_len - 2
        )
        return self.rel_pos_embed(distance_mat_clipped)

# --- Transformer Block ---
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.dk = d_model // heads
        self.heads = heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rel_pos_embed = RelativePositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, padding_mask=None):
        B, T, _ = x.size()
        q = self.q_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.dk)
        
        # Add relative positions
        rel_pos = self.rel_pos_embed(T, T).to(x.device)
        rel_pos = rel_pos.view(T, T, self.heads, self.dk).permute(2, 0, 1, 3)
        scores += torch.einsum("bhtd,htsd->bhts", q, rel_pos)
        
        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(1), float('-inf'))
        
        # Apply padding mask if provided
        if padding_mask is not None:
            padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)
            scores = scores.masked_fill(~padding_mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        
        # Residual connections
        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x

# --- Generator Model ---
class CausalTransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.ModuleList([
            CausalTransformerBlock(d_model, heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        print(f"Model vocab size: {vocab_size}")

    def forward(self, x, padding_mask=None):
        x = self.embedding(x)
        for layer in self.transformer:
            x = layer(x, padding_mask)
        return self.lm_head(x)

# --- Load Data ---
try:
    train_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\train_set.csv")
    test_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\val_set.csv")
    
    # Handle missing data
    train_df = train_df.dropna(subset=['question', 'answer'])
    test_df = test_df.dropna(subset=['question', 'answer'])
    
    print(f"Training samples: {len(train_df)} | Test samples: {len(test_df)}")
    
    # Initialize datasets
    train_dataset = QA_Dataset(
        train_df["question"].tolist(),
        train_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    test_dataset = QA_Dataset(
        test_df["question"].tolist(),
        test_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy datasets if files not found
    class DummyDataset(Dataset):
        def __len__(self): return 100
        def __getitem__(self, idx):
            return torch.zeros(MAX_LEN, dtype=torch.long), \
                   torch.zeros(MAX_LEN, dtype=torch.long), \
                   torch.zeros(MAX_LEN, dtype=torch.long)
    
    train_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)
    test_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)

# --- Initialize Model ---
model = CausalTransformerGenerator(VOCAB_SIZE, D_MODEL, HEADS, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore masked tokens

# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, token_type_ids, labels = batch
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # Create padding mask
        padding_mask = (input_ids != tokenizer.pad_token_id).float().to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(input_ids, padding_mask=padding_mask)
        
        # Calculate loss
        loss = criterion(outputs.view(-1, VOCAB_SIZE), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

# --- Generation Function ---
def generate_answer(question, max_len=30, temperature=1.0):
    model.eval()
    # Tokenize question
    inputs = tokenizer.encode_plus(
        question,
        max_length=MAX_LEN,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    
    # Generate tokens
    for _ in range(max_len):
        with torch.no_grad():
            outputs = model(input_ids, padding_mask=attention_mask)
        
        # Get last token logits
        logits = outputs[0, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        
        # Stop if [SEP] is generated
        if next_token.item() == tokenizer.sep_token_id:
            break
            
        # Update inputs
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        attention_mask = torch.cat([
            attention_mask, 
            torch.ones(1, 1, dtype=attention_mask.dtype, device=DEVICE)
        ], dim=1)
        
        # Stop if max length reached
        if input_ids.shape[1] >= MAX_LEN:
            break
    
    # Decode and clean output
    output_text = tokenizer.decode(
        input_ids[0], 
        skip_special_tokens=True
    )
    # Remove input question if present
    if output_text.startswith(question):
        output_text = output_text[len(question):].strip()
    return output_text

# --- Test Generation ---
print("\nTest Generation:")
test_questions = [
    "What is deep learning?",
    "How does photosynthesis work?",
    "What causes seasons on Earth?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {generate_answer(q)}")

Using device: cuda
Tokenizer vocab size: 30522
Training samples: 3775 | Test samples: 944
Model vocab size: 30522


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


TypeError: ~ (operator.invert) is only implemented on integer and Boolean-type tensors

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel

# --- Hyperparameters ---
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5  # Reduced for testing
D_MODEL = 128
HEADS = 4
NUM_LAYERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Load standard BERT tokenizer ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

# --- Simplified Dataset ---
class QA_Dataset(Dataset):
    def __init__(self, questions, answers, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.questions = questions
        self.answers = answers
        
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        question = str(self.questions[idx])
        answer = str(self.answers[idx])
        
        # Tokenize with question-answer format
        encoding = self.tokenizer(
            question,
            answer,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        token_type_ids = encoding['token_type_ids'].squeeze(0)
        
        # Create labels (shift next-token prediction)
        labels = input_ids.clone()
        # Only compute loss on answer part (token_type_ids=1)
        labels[token_type_ids == 0] = -100
        # Shift labels for next-token prediction
        labels[:-1] = labels[1:].clone()
        labels[-1] = -100
        
        return input_ids, token_type_ids, labels

# --- Fixed Positional Embedding ---
class RelativePositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.rel_pos_embed = nn.Embedding(2 * max_len - 1, d_model)

    def forward(self, qlen, klen):
        device = self.rel_pos_embed.weight.device
        range_vec = torch.arange(qlen, device=device)
        distance_mat = range_vec[:, None] - range_vec[None, :]
        distance_mat_clipped = torch.clamp(
            distance_mat + self.max_len - 1, 
            min=0, 
            max=2 * self.max_len - 2
        )
        return self.rel_pos_embed(distance_mat_clipped)

# --- Transformer Block ---
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.dk = d_model // heads
        self.heads = heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rel_pos_embed = RelativePositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, padding_mask=None):
        B, T, _ = x.size()
        q = self.q_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.dk)
        
        # Add relative positions
        rel_pos = self.rel_pos_embed(T, T).to(x.device)
        rel_pos = rel_pos.view(T, T, self.heads, self.dk).permute(2, 0, 1, 3)
        scores += torch.einsum("bhtd,htsd->bhts", q, rel_pos)
        
        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(1), float('-inf'))
        
        # Apply padding mask if provided
        if padding_mask is not None:
            # Convert to boolean mask: True means valid token, False means padding
            padding_mask = padding_mask.bool()
            # Reshape for attention scores: [B, 1, 1, T]
            padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)
            # Expand to match scores dimensions: [B, heads, T, T]
            padding_mask = padding_mask.expand(-1, self.heads, T, -1)
            # Apply padding mask to scores
            scores = scores.masked_fill(~padding_mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        
        # Residual connections
        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x

# --- Generator Model ---
class CausalTransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.ModuleList([
            CausalTransformerBlock(d_model, heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        print(f"Model vocab size: {vocab_size}")

    def forward(self, x, padding_mask=None):
        x = self.embedding(x)
        for layer in self.transformer:
            x = layer(x, padding_mask)
        return self.lm_head(x)

# --- Load Data ---
try:
    train_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\train_set.csv")
    test_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\val_set.csv")
    
    # Handle missing data
    train_df = train_df.dropna(subset=['question', 'answer'])
    test_df = test_df.dropna(subset=['question', 'answer'])
    
    print(f"Training samples: {len(train_df)} | Test samples: {len(test_df)}")
    
    # Initialize datasets
    train_dataset = QA_Dataset(
        train_df["question"].tolist(),
        train_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    test_dataset = QA_Dataset(
        test_df["question"].tolist(),
        test_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Creating dummy datasets...")
    # Create dummy datasets if files not found
    class DummyDataset(Dataset):
        def __len__(self): return 100
        def __getitem__(self, idx):
            return (torch.zeros(MAX_LEN, dtype=torch.long), 
                    torch.zeros(MAX_LEN, dtype=torch.long), 
                    torch.zeros(MAX_LEN, dtype=torch.long))
    
    train_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)
    test_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)

# --- Initialize Model ---
model = CausalTransformerGenerator(VOCAB_SIZE, D_MODEL, HEADS, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore masked tokens

# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, _, labels = batch  # We don't need token_type_ids here
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # Create boolean padding mask (True for real tokens, False for padding)
        padding_mask = (input_ids != tokenizer.pad_token_id).to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(input_ids, padding_mask=padding_mask)
        
        # Calculate loss
        loss = criterion(outputs.view(-1, VOCAB_SIZE), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

# --- Generation Function ---
def generate_answer(question, max_len=30, temperature=1.0):
    model.eval()
    # Tokenize question
    inputs = tokenizer.encode_plus(
        question,
        max_length=MAX_LEN,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    
    # Generate tokens
    for _ in range(max_len):
        with torch.no_grad():
            outputs = model(input_ids, padding_mask=attention_mask)
        
        # Get last token logits
        logits = outputs[0, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        
        # Stop if [SEP] is generated
        if next_token.item() == tokenizer.sep_token_id:
            break
            
        # Update inputs
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        attention_mask = torch.cat([
            attention_mask, 
            torch.ones(1, 1, dtype=attention_mask.dtype, device=DEVICE)
        ], dim=1)
        
        # Stop if max length reached
        if input_ids.shape[1] >= MAX_LEN:
            break
    
    # Decode and clean output
    output_text = tokenizer.decode(
        input_ids[0], 
        skip_special_tokens=True
    )
    # Remove input question if present
    if output_text.startswith(question):
        output_text = output_text[len(question):].strip()
    return output_text

# --- Test Generation ---
print("\nTest Generation:")
test_questions = [
    "What is deep learning?",
    "How does photosynthesis work?",
    "What causes seasons on Earth?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {generate_answer(q)}")

Using device: cuda
Tokenizer vocab size: 30522


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Training samples: 3775 | Test samples: 944
Model vocab size: 30522


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 1/5 | Loss: 9.5106


...

Epoch 3/5 | Loss: 7.5708


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 4/5 | Loss: 7.4261


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 5/5 | Loss: 7.3466

Test Generation:

Q: What is deep learning?
A: what is deep learning? we role a to cyop nu

Q: How does photosynthesis work?
A: how does photosynthesis work? invasions after prors marsiaitategging

Q: What causes seasons on Earth?
A: what causes seasons on earth?pressingท diseaseome pathogenzine fallingco upon a dnal myomic brain developed abnormal30 in childrenl characterizediraoder million of the localized dna development


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel

# --- Hyperparameters ---
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 25  # Reduced for testing
D_MODEL = 128
HEADS = 4
NUM_LAYERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Load standard BERT tokenizer ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

# --- Simplified Dataset ---
class QA_Dataset(Dataset):
    def __init__(self, questions, answers, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.questions = questions
        self.answers = answers
        
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        question = str(self.questions[idx])
        answer = str(self.answers[idx])
        
        # Tokenize with question-answer format
        encoding = self.tokenizer(
            question,
            answer,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        token_type_ids = encoding['token_type_ids'].squeeze(0)
        
        # Create labels (shift next-token prediction)
        labels = input_ids.clone()
        # Only compute loss on answer part (token_type_ids=1)
        labels[token_type_ids == 0] = -100
        # Shift labels for next-token prediction
        labels[:-1] = labels[1:].clone()
        labels[-1] = -100
        
        return input_ids, token_type_ids, labels

# --- Fixed Positional Embedding ---
class RelativePositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.rel_pos_embed = nn.Embedding(2 * max_len - 1, d_model)

    def forward(self, qlen, klen):
        device = self.rel_pos_embed.weight.device
        range_vec = torch.arange(qlen, device=device)
        distance_mat = range_vec[:, None] - range_vec[None, :]
        distance_mat_clipped = torch.clamp(
            distance_mat + self.max_len - 1, 
            min=0, 
            max=2 * self.max_len - 2
        )
        return self.rel_pos_embed(distance_mat_clipped)

# --- Transformer Block ---
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.dk = d_model // heads
        self.heads = heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rel_pos_embed = RelativePositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, padding_mask=None):
        B, T, _ = x.size()
        q = self.q_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.dk).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.dk)
        
        # Add relative positions
        rel_pos = self.rel_pos_embed(T, T).to(x.device)
        rel_pos = rel_pos.view(T, T, self.heads, self.dk).permute(2, 0, 1, 3)
        scores += torch.einsum("bhtd,htsd->bhts", q, rel_pos)
        
        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(1), float('-inf'))
        
        # Apply padding mask if provided
        if padding_mask is not None:
            # Convert to boolean mask: True means valid token, False means padding
            padding_mask = padding_mask.bool()
            # Reshape for attention scores: [B, 1, 1, T]
            padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)
            # Expand to match scores dimensions: [B, heads, T, T]
            padding_mask = padding_mask.expand(-1, self.heads, T, -1)
            # Apply padding mask to scores
            scores = scores.masked_fill(~padding_mask, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        
        # Residual connections
        x = self.ln1(x + out)
        x = self.ln2(x + self.ff(x))
        return x

# --- Generator Model ---
class CausalTransformerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.ModuleList([
            CausalTransformerBlock(d_model, heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        print(f"Model vocab size: {vocab_size}")

    def forward(self, x, padding_mask=None):
        x = self.embedding(x)
        for layer in self.transformer:
            x = layer(x, padding_mask)
        return self.lm_head(x)

# --- Load Data ---
try:
    train_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\train_set.csv")
    test_df = pd.read_csv("C:\\PG AI and DS\\Semester 3\\NLP AISC2009\\val_set.csv")
    
    # Handle missing data
    train_df = train_df.dropna(subset=['question', 'answer'])
    test_df = test_df.dropna(subset=['question', 'answer'])
    
    print(f"Training samples: {len(train_df)} | Test samples: {len(test_df)}")
    
    # Initialize datasets
    train_dataset = QA_Dataset(
        train_df["question"].tolist(),
        train_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    test_dataset = QA_Dataset(
        test_df["question"].tolist(),
        test_df["answer"].tolist(),
        tokenizer,
        MAX_LEN
    )
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Creating dummy datasets...")
    # Create dummy datasets if files not found
    class DummyDataset(Dataset):
        def __len__(self): return 100
        def __getitem__(self, idx):
            return (torch.zeros(MAX_LEN, dtype=torch.long), 
                    torch.zeros(MAX_LEN, dtype=torch.long), 
                    torch.zeros(MAX_LEN, dtype=torch.long))
    
    train_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)
    test_loader = DataLoader(DummyDataset(), batch_size=BATCH_SIZE)

# --- Initialize Model ---
model = CausalTransformerGenerator(VOCAB_SIZE, D_MODEL, HEADS, NUM_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore masked tokens

# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, _, labels = batch  # We don't need token_type_ids here
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # Create boolean padding mask (True for real tokens, False for padding)
        padding_mask = (input_ids != tokenizer.pad_token_id).to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(input_ids, padding_mask=padding_mask)
        
        # Calculate loss
        loss = criterion(outputs.view(-1, VOCAB_SIZE), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

# --- Generation Function ---
def generate_answer(question, max_len=30, temperature=1.0):
    model.eval()
    # Tokenize question
    inputs = tokenizer.encode_plus(
        question,
        max_length=MAX_LEN,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)
    
    # Generate tokens
    for _ in range(max_len):
        with torch.no_grad():
            outputs = model(input_ids, padding_mask=attention_mask)
        
        # Get last token logits
        logits = outputs[0, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        
        # Stop if [SEP] is generated
        if next_token.item() == tokenizer.sep_token_id:
            break
            
        # Update inputs
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        attention_mask = torch.cat([
            attention_mask, 
            torch.ones(1, 1, dtype=attention_mask.dtype, device=DEVICE)
        ], dim=1)
        
        # Stop if max length reached
        if input_ids.shape[1] >= MAX_LEN:
            break
    
    # Decode and clean output
    output_text = tokenizer.decode(
        input_ids[0], 
        skip_special_tokens=True
    )
    # Remove input question if present
    if output_text.startswith(question):
        output_text = output_text[len(question):].strip()
    return output_text

# --- Test Generation ---
print("\nTest Generation:")
test_questions = [
    "What is deep learning?",
    "How does photosynthesis work?",
    "What causes seasons on Earth?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {generate_answer(q)}")

Using device: cuda
Tokenizer vocab size: 30522
Training samples: 3775 | Test samples: 944
Model vocab size: 30522


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 1/25 | Loss: 9.5501


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 2/25 | Loss: 8.0769


...

Epoch 23/25 | Loss: 5.6478


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 24/25 | Loss: 5.5808


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Epoch 25/25 | Loss: 5.5191

Test Generation:

Q: What is deep learning?
A: what is deep learning? commonly clear book 401 numerous it is a hands test pal is being applications repertoire and propc family oract characterized by neing in metabol ligas

Q: How does photosynthesis work?
A: how does photosynthesis work? inheritance to response to cell regulate and2 resulting in stimulate seizuresgeneous re 135 metabolic star is a macro of histone tails

Q: What causes seasons on Earth?
A: what causes seasons on earth? disease analyzing crispcorp abundance genesum used for process of enhancers hamrs to admission in these abundance of robinbuase region tostand through the atp
